In [1]:
import os
import boto3
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time
import pandas as pd
import pickle
from tqdm import tqdm

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# function name
str_function_name = 'genxi-parse-payloads'

Project: 20231010-gen-xii


### Functions

In [3]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

### 1. Create container

### Create ```Dockerfile```

In [4]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy toolbox
COPY gopfsrisk_toolbox ${LAMBDA_TASK_ROOT}/gopfsrisk_toolbox

# copy parser
COPY cls_parse_payload_with_aa.pkl ${LAMBDA_TASK_ROOT}

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd
import numpy as np
from datetime import datetime
import pickle
import json

# lambda handler
def lambda_handler(event, context):
    # get the input
    try:
        int_rows_to_parse = int(event['row'])
    except:
        int_rows_to_parse = 1
    print(f'Parsing rows: {int_rows_to_parse}')
    
    # constants
    str_project = '20231010-gen-xii'
    str_task = 'ad_hoc'
    str_subtask = 'gen_11_payload_parsing'
    str_final_task = 'parsed_payloads'
    
    # get today's date
    str_date_today = datetime.today().strftime('%Y%m%d')
    
    # load requests
    print('Loading requests...')
    str_filename = f'df_rows_{int_rows_to_parse}.gzip'
    str_uri = f's3://{str_project}/{str_task}/{str_subtask}/days/{str_date_today}/split_payloads/{str_filename}'
    df = pd.read_parquet(str_uri)
    print(f'There are {df.shape[0]} requests for this lambda function to parse')
    
    # load parser
    print('Loading parser...')
    str_filename = 'cls_parse_payload_with_aa.pkl'
    str_local_path = f'./{str_filename}'
    cls_parser = pickle.load(open(str_local_path, 'rb'))
    
    # parse (get income, LN, and TU)
    print('Parsing requests...')
    list_df_tmp = []
    for a, str_request in enumerate(df['REQUEST_JSON']):
        # get bigAccountId
        int_bigaccountid = df['ACCOUNTID'].iloc[a]
        # get applicationdate
        dtm_app_date = df['REQUEST_DATETIME'].iloc[a]
    
        # convert string request to dict
        dict_json_request = json.loads(str_request)
        # generate predictions
        cls_parser.generate_predictions(dict_json_request)
        
        # get list of PD annd LGD
        # get pd
        list_yhat_pd = cls_parser.y_hat_pd
        # get lgd
        list_yhat_lgd = cls_parser.y_hat_lgd
    
        # get values of ecnl and modified ecnl
        # get ecnl
        yhat_ecnl = cls_parser.y_hat_pd_x_lgd
        # get ecnl mod
        yhat_ecnl_mod = cls_parser.y_hat_pd_x_lgd_mod
        
        # logic to prevent length mismatch of columns
        if len(list_yhat_pd) == 1:
            list_account_id = [int_bigaccountid]
            list_app_date = [dtm_app_date]
            list_yhat_ecnl = [yhat_ecnl]
            list_yhat_ecnl_mod = [yhat_ecnl_mod]
            list_bitdebtor = [1]
        else:
            list_account_id = [int_bigaccountid, int_bigaccountid]
            list_app_date = [dtm_app_date, dtm_app_date]
            list_yhat_ecnl = [yhat_ecnl, yhat_ecnl]
            list_yhat_ecnl_mod = [yhat_ecnl_mod, yhat_ecnl_mod]
            list_bitdebtor = [1, 0]
        
        # create df
        df_tmp = pd.DataFrame({
            'bigAccountId': list_account_id,
            'ApplicationDate': list_app_date,
            'list_yhat_pd': list_yhat_pd,
            'list_yhat_lgd': list_yhat_lgd,
            'list_yhat_ecnl': list_yhat_ecnl,
            'list_yhat_ecnl_mod': list_yhat_ecnl_mod,
            'BITDEBTOR': list_bitdebtor,
        })
        
        # append
        list_df_tmp.append(df_tmp)
    
    # concat
    print('Concatenating data...')
    df = pd.concat(list_df_tmp)
    
    # save memeory
    del list_df_tmp
    
    # write to s3 as parquet
    print('Writing raw data to s3...')
    # set nonnumeric to string
    for col in df.columns:
        # if not numeric
        if df[col].dtype not in ['int64', 'float64']:
            # set as string
            df[col] = df[col].astype(str)
        else:
            pass
    # write to gzip
    str_filename = f'df_{int_rows_to_parse}.gzip'
    str_uri = f's3://{str_project}/{str_task}/{str_subtask}/days/{str_date_today}/{str_final_task}/{str_filename}'
    df.to_parquet(str_uri, compression='gzip')

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxi-parse-payloads

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  15.28MB
Step 1/8 : FROM public.ecr.aws/lambda/python:3.8
 ---> 3cd81ffec4d9
Step 2/8 : RUN pip install --upgrade pip
 ---> Using cache
 ---> f19e3966cd6e
Step 3/8 : COPY requirements.txt  .
 ---> 433d5669dbf3
Step 4/8 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Running in 04607c1090a5
INFO: pip is looking at multiple versions of fastparquet to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of fastparquet to determine which version is compatible with other requirements. This could take a while.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.1/283.1 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxi-parse-payloads' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxi-parse-payloads]
4c84fd0875f3: Preparing
6ebe7cf201df: Preparing
e6b7639ea3fd: Preparing
56dffb860b2b: Preparing
64c5b246b950: Preparing
417f19c472ac: Preparing
e92756f7b561: Preparing
4fe51bf0bf5c: Preparing
fbbd8c1e2ec1: Preparing
fe2359fe88f2: Preparing
e703f2e518cc: Preparing
97a787951169: Preparing
417f19c472ac: Waiting
e92756f7b561: Waiting
4fe51bf0bf5c: Waiting
fbbd8c1e2ec1: Waiting
fe2359fe88f2: Waiting
e703f2e518cc: Waiting
97a787951169: Waiting
64c5b246b950: Pushed
4c84fd0875f3: Pushed
417f19c472ac: Layer already exists
e6b7639ea3fd: Pushed
e92756f7b561: Layer already exists
4fe51bf0bf5c: Layer already exists
fe2359fe88f2: Layer already exists
e703f2e518cc: Layer already exists
fbbd8c1e2ec1: Layer already exists
97a787951169: Layer already exists
6ebe7cf201df: Pushed
56dffb860b2b: Pushed
latest: digest: sha256:d9feebb454442346b9ff3ed6237120fcd581486826ed03d95d7684bbb7b96b50 size: 2840


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 30 Apr 2024 20:07:39 GMT',
                                      'x-amzn-requestid': 'a688ebcb-a009-4417-8249-a42201141921'},
                      'HTTPStatusCode': 204,
                      'RequestId': 'a688ebcb-a009-4417-8249-a42201141921',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=900, # 15 minutes is maximum
    MemorySize=1000, # 1000 mb == 1 gb
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 1000, # 1000 mb == 1 gb
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': 'd9feebb454442346b9ff3ed6237120fcd581486826ed03d95d7684bbb7b96b50',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 1000},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxi-parse-payloads',
 'FunctionName': 'genxi-parse-payloads',
 'LastModified': '2024-04-30T20:07:39.758+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxi-parse-payloads'},
 'MemorySize': 1000,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1195',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 30 Apr 2024 20:07:40 GMT',
                                      'x-amzn-requestid': 'f8e105af-8f1c-49e1-a861-b3e8bc199925'},
                      'HTTPStatusCode': 201,
                      'RequestId': 'f8e105af-8f1c-49e1-a

### Clean-up

In [11]:
list_str_filename = ['Dockerfile', 'lambda_function.py']

for str_file in list_str_filename:
    os.remove(str_file)